# Skin Disease Image Classification with YOLOv8

This notebook trains a **4-class image-classification model** for the image folders in the Kaggle dataset:

- Chickenpox
- Measles
- Monkeypox
- Normal

**Important:** This dataset contains class folders, not object-detection labels or bounding boxes. Therefore, this project uses `yolov8n-cls.pt` (classification), not `yolov8n.pt` (object detection). It does not use `data.yaml`.

### Dataset selection

The default source is `Dataset_Gabungan`, which appears to contain images combined from Dataset 1 and Dataset 2 (the filenames include `ds1_` and `ds2_`). Use only one source folder at a time so the original datasets are not accidentally included multiple times. You can change the source in the configuration cell.

Run the cells from top to bottom in Google Colab. If the dataset is not already present in the runtime, the notebook will ask you to upload the Kaggle ZIP file.

**Medical-use limitation:** This is an educational image-classification project. Its predictions are not a medical diagnosis and must not be used to decide treatment or delay professional medical care.

In [ ]:
# Install YOLO and the libraries used for dataset splitting and image handling.
!pip -q install -U ultralytics scikit-learn pillow matplotlib

In [ ]:
import os
import random
import shutil
import zipfile
import hashlib
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.model_selection import train_test_split
from IPython.display import display

import torch
from ultralytics import YOLO

# ------------------------- PROJECT SETTINGS -------------------------
# Choose ONE source folder: "Dataset_Gabungan", "Dataset_1", or "Dataset_2".
SOURCE_FOLDER_NAME = "Dataset_Gabungan"

# Class folder names expected inside the selected source folder.
CLASS_NAMES = ["Chickenpox", "Measles", "Monkeypox", "Normal"]

# Reproducibility and training configuration.
RANDOM_SEED = 42
TRAIN_RATIO = 0.70
VALIDATION_RATIO = 0.15
TEST_RATIO = 0.15
EPOCHS = 20
IMAGE_SIZE = 224
BATCH_SIZE = 16

# Paths used by this notebook.
CONTENT_DIR = Path("/content")
PREPARED_DATASET_DIR = CONTENT_DIR / "disease_classification_dataset"
RUN_PROJECT_DIR = CONTENT_DIR / "runs" / "classify"
RUN_NAME = "disease_classification"
BEST_MODEL_PATH = RUN_PROJECT_DIR / RUN_NAME / "weights" / "best.pt"

# Supported image extensions.
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

DEVICE = 0 if torch.cuda.is_available() else "cpu"
print("PyTorch:", torch.__version__)
print("Ultralytics:", __import__("ultralytics").__version__)
print("Training device:", "GPU 0" if DEVICE == 0 else "CPU")
print("Selected source folder:", SOURCE_FOLDER_NAME)

In [ ]:
from google.colab import files


def is_classification_folder(folder: Path) -> bool:
    """Return True when folder directly contains all expected class folders."""
    if not folder.is_dir():
        return False
    child_names = {p.name.casefold() for p in folder.iterdir() if p.is_dir()}
    return all(class_name.casefold() in child_names for class_name in CLASS_NAMES)


def find_source_folder(search_root: Path):
    """Find the selected source folder anywhere below search_root."""
    for candidate in search_root.rglob(SOURCE_FOLDER_NAME):
        if is_classification_folder(candidate):
            return candidate
    return None


# First use a dataset that is already extracted in the current Colab runtime.
source_dir = find_source_folder(CONTENT_DIR)

if source_dir is None:
    print(f"'{SOURCE_FOLDER_NAME}' was not found in /content.")
    print("Upload the Kaggle ZIP file that contains Dataset_1, Dataset_2, and Dataset_Gabungan.")
    uploaded_files = files.upload()

    zip_names = [name for name in uploaded_files if name.lower().endswith(".zip")]
    if not zip_names:
        raise FileNotFoundError("No ZIP file was uploaded. Please upload your Kaggle dataset ZIP file.")

    extraction_dir = CONTENT_DIR / "kaggle_dataset_upload"
    extraction_dir.mkdir(parents=True, exist_ok=True)

    for zip_name in zip_names:
        with zipfile.ZipFile(zip_name, "r") as zip_ref:
            zip_ref.extractall(extraction_dir)
        print("Extracted:", zip_name)

    source_dir = find_source_folder(extraction_dir)

if source_dir is None:
    raise FileNotFoundError(
        f"Could not find a valid '{SOURCE_FOLDER_NAME}' folder containing these class folders: {CLASS_NAMES}. "
        "Check the ZIP file and the SOURCE_FOLDER_NAME setting."
    )

print("\nDataset source found:", source_dir)
print("Class folders:", [p.name for p in source_dir.iterdir() if p.is_dir()])

In [ ]:
# Inspect the selected source and count readable image files in each class.
class_counts = {}
example_images = {}

for class_name in CLASS_NAMES:
    class_dir = next(
        (p for p in source_dir.iterdir() if p.is_dir() and p.name.casefold() == class_name.casefold()),
        None
    )
    if class_dir is None:
        raise FileNotFoundError(f"Missing class folder: {class_name}")

    image_paths = [p for p in class_dir.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS]
    class_counts[class_name] = len(image_paths)
    example_images[class_name] = image_paths[:1]

count_table = pd.DataFrame(
    [{"Class": class_name, "Image files": class_counts[class_name]} for class_name in CLASS_NAMES]
)
display(count_table)

if any(count == 0 for count in class_counts.values()):
    raise ValueError("At least one class contains no supported images. Check the selected dataset folder.")

# Display one example image from each class.
fig, axes = plt.subplots(1, len(CLASS_NAMES), figsize=(16, 4))
for ax, class_name in zip(axes, CLASS_NAMES):
    image_path = example_images[class_name][0]
    with Image.open(image_path) as image:
        ax.imshow(image.convert("RGB"))
    ax.set_title(class_name)
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Build a clean image list. This validates image files and removes exact duplicate files
# (based on SHA-256) so duplicate copies do not appear in both training and test sets.
records = []
seen_hashes = {}
invalid_images = []
duplicate_images = []
conflicting_duplicates = []

for class_name in CLASS_NAMES:
    class_dir = next(
        p for p in source_dir.iterdir()
        if p.is_dir() and p.name.casefold() == class_name.casefold()
    )

    for image_path in sorted(class_dir.rglob("*")):
        if not image_path.is_file() or image_path.suffix.lower() not in IMAGE_EXTENSIONS:
            continue

        try:
            # Verify the file can be decoded as an image.
            with Image.open(image_path) as image:
                image.verify()

            file_hash = hashlib.sha256(image_path.read_bytes()).hexdigest()

            if file_hash in seen_hashes:
                previous_class = seen_hashes[file_hash]
                if previous_class == class_name:
                    duplicate_images.append(str(image_path))
                else:
                    conflicting_duplicates.append((str(image_path), previous_class, class_name))
                continue

            seen_hashes[file_hash] = class_name
            records.append({"path": image_path, "label": class_name})

        except Exception as error:
            invalid_images.append((str(image_path), str(error)))

print("Unique valid images:", len(records))
print("Exact duplicate files skipped:", len(duplicate_images))
print("Unreadable/corrupt images skipped:", len(invalid_images))
print("Exact duplicates found under conflicting labels:", len(conflicting_duplicates))

if conflicting_duplicates:
    print("Examples of conflicting duplicate labels:")
    for item in conflicting_duplicates[:10]:
        print(item)
    raise ValueError(
        "The same exact image file appears under different class labels. Resolve these label conflicts before training."
    )

clean_counts = Counter(record["label"] for record in records)
print("\nUnique image count by class:")
for class_name in CLASS_NAMES:
    print(f"{class_name}: {clean_counts[class_name]}")

if any(clean_counts[class_name] < 3 for class_name in CLASS_NAMES):
    raise ValueError("Each class needs at least 3 unique valid images for a stratified train/validation/test split.")

In [ ]:
# Split images with stratification so each split contains all four classes.
# 70% train, 15% validation, 15% test.
all_paths = [str(record["path"]) for record in records]
all_labels = [record["label"] for record in records]

train_paths, temporary_paths, train_labels, temporary_labels = train_test_split(
    all_paths,
    all_labels,
    test_size=(VALIDATION_RATIO + TEST_RATIO),
    random_state=RANDOM_SEED,
    stratify=all_labels
)

relative_test_size = TEST_RATIO / (VALIDATION_RATIO + TEST_RATIO)
validation_paths, test_paths, validation_labels, test_labels = train_test_split(
    temporary_paths,
    temporary_labels,
    test_size=relative_test_size,
    random_state=RANDOM_SEED,
    stratify=temporary_labels
)

splits = {
    "train": list(zip(train_paths, train_labels)),
    "val": list(zip(validation_paths, validation_labels)),
    "test": list(zip(test_paths, test_labels)),
}

# Re-create the prepared dataset folder each time this cell is run.
if PREPARED_DATASET_DIR.exists():
    shutil.rmtree(PREPARED_DATASET_DIR)

for split_name, split_records in splits.items():
    for image_path, class_name in split_records:
        destination_dir = PREPARED_DATASET_DIR / split_name / class_name
        destination_dir.mkdir(parents=True, exist_ok=True)
        shutil.copy2(image_path, destination_dir / Path(image_path).name)

print("Prepared YOLO classification dataset:", PREPARED_DATASET_DIR)
print("\nSplit summary:")
for split_name, split_records in splits.items():
    counts = Counter(label for _, label in split_records)
    print(f"{split_name.upper():5} | total: {len(split_records)} | " + " | ".join(f"{name}: {counts[name]}" for name in CLASS_NAMES))

print("\nFolder structure created:")
for split_name in ["train", "val", "test"]:
    print(f"{split_name}/")
    for class_name in CLASS_NAMES:
        folder = PREPARED_DATASET_DIR / split_name / class_name
        print(f"    {class_name}/ ({len(list(folder.glob('*')))} images)")

## Train the classification model

The next cell uses the small YOLOv8 classification model (`yolov8n-cls.pt`). It automatically selects a Colab GPU when available; otherwise it trains on CPU, which can be slower.

In [ ]:
# Train YOLOv8 image classification model.
model = YOLO("yolov8n-cls.pt")

train_results = model.train(
    data=str(PREPARED_DATASET_DIR),
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    workers=2,
    device=DEVICE,
    project=str(RUN_PROJECT_DIR),
    name=RUN_NAME,
    exist_ok=True,
    patience=5,
    plots=True,
    verbose=True
)

print("\nTraining finished.")
print("Expected best model path:", BEST_MODEL_PATH)
print("Best model file exists:", BEST_MODEL_PATH.exists())

In [ ]:
# Load the best checkpoint saved during training.
if not BEST_MODEL_PATH.exists():
    raise FileNotFoundError(
        f"Best checkpoint not found at {BEST_MODEL_PATH}. Check the training output above."
    )

best_model = YOLO(str(BEST_MODEL_PATH))
print("Best classification model loaded successfully.")
print("Class names:", best_model.names)

## Evaluate on the held-out test set

This evaluates images that were not used for training or validation. The reported top-1 accuracy is the fraction for which the model's highest-confidence class matches the folder label. Top-5 accuracy is also reported by Ultralytics; with four classes, it is less informative than top-1.

In [ ]:
# Evaluate using the test split (not the validation split).
test_metrics = best_model.val(
    data=str(PREPARED_DATASET_DIR),
    split="test",
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    plots=True
)

print("\nTEST RESULTS")
print(f"Top-1 accuracy: {test_metrics.top1:.4f}")
print(f"Top-5 accuracy: {test_metrics.top5:.4f}")

In [ ]:
# Display the training curves and confusion matrix when those plot files were generated.
from IPython.display import Image as NotebookImage

results_plot = RUN_PROJECT_DIR / RUN_NAME / "results.png"
confusion_plot = RUN_PROJECT_DIR / "val" / "confusion_matrix.png"

if results_plot.exists():
    print("Training results:")
    display(NotebookImage(filename=str(results_plot)))
else:
    print("Training results plot was not found:", results_plot)

# Ultralytics normally saves validation plots in a separate validation run folder.
# Search the classify runs folder in case the generated validation folder has a different name.
confusion_candidates = list(RUN_PROJECT_DIR.rglob("confusion_matrix.png"))
if confusion_candidates:
    print("Confusion matrix:")
    display(NotebookImage(filename=str(confusion_candidates[-1])))
else:
    print("No confusion_matrix.png was generated in the run folder.")

## Predict a new image

Upload one or more JPG/PNG images. The notebook will show the predicted class and confidence score for each image.

In [ ]:
from google.colab import files

prediction_uploads = files.upload()

if not prediction_uploads:
    print("No prediction images were uploaded.")
else:
    for image_name in prediction_uploads:
        image_path = Path(image_name)

        if image_path.suffix.lower() not in IMAGE_EXTENSIONS:
            print(f"Skipping unsupported file: {image_name}")
            continue

        prediction_results = best_model.predict(
            source=str(image_path),
            imgsz=IMAGE_SIZE,
            device=DEVICE,
            verbose=False
        )

        prediction = prediction_results[0]
        predicted_index = int(prediction.probs.top1)
        predicted_class = best_model.names[predicted_index]
        confidence = float(prediction.probs.top1conf)

        with Image.open(image_path) as image:
            plt.figure(figsize=(5, 5))
            plt.imshow(image.convert("RGB"))
            plt.axis("off")
            plt.title(f"Prediction: {predicted_class} ({confidence:.2%})")
            plt.show()

        print(f"Image: {image_name}")
        print(f"Predicted class: {predicted_class}")
        print(f"Confidence: {confidence:.2%}")
        print("Reminder: This is an experimental image-classification result, not a medical diagnosis.")
        print("-" * 60)

## Download the trained model

This downloads the best checkpoint (`best.pt`) to your computer. Keep this file to reuse the trained model without training again.

In [ ]:
from google.colab import files

if BEST_MODEL_PATH.exists():
    files.download(str(BEST_MODEL_PATH))
else:
    raise FileNotFoundError(f"Model file does not exist: {BEST_MODEL_PATH}")

## Notes

- To use Dataset 1 only, change `SOURCE_FOLDER_NAME` to `Dataset_1` in the configuration cell.
- To use Dataset 2 only, change it to `Dataset_2`.
- The default is `Dataset_Gabungan`; do not concatenate the three source folders because the combined folder appears to contain images from the two original datasets already.
- If you change `SOURCE_FOLDER_NAME`, rerun the notebook from the source-detection cell onward. For a clean run, use **Runtime → Restart session** and run all cells.
- Training on CPU may take considerably longer than training on a GPU.